## LC 139 — Word Break (Morse)

**Classical 139:** You have a string of letters and a dictionary of English words. Can you split the whole string into words from the dictionary (reuse allowed)?

**This variant:** The **input** `s` is already a Morse **bitstring** (only `'.'` and `'-'`), with no gaps between “letters.” Each word in `wordDict` is plain lowercase text; you **encode** it to Morse by concatenating each letter’s ITU code (use `MORSE_CODE` in the next cell). For example `"we"` → `.--` + `.` = `.--.`, and `"s"` → `...`.

**Task:** Return `True` iff `s` equals a concatenation of **one or more** such encodings (same word can appear multiple times). Equivalently: segment `s` into chunks where each chunk is exactly `encode(w)` for some `w` in `wordDict`.

**Edge cases:** Empty `s` → `True`. Non-empty `s` with empty `wordDict` → `False`.

**Examples (encode with the table below):**
- `s = ".--...."`, `wordDict = ["we", "s"]` → `encode("we")` + `encode("s")` = `.--.` + `...` → **True**.
- `s = "...."`, `wordDict = ["h"]` only → you only have `....`; **True**. With `["ee"]` you have `..` twice → **True** for `"...."` = `..` + `..`.

`def wordBreakMorse(s: str, wordDict) -> bool`


In [2]:
MORSE_CODE = {
    'a': '.-',
    'b': '-...',
    'c': '-.-.',
    'd': '-..',
    'e': '.',
    'f': '..-.',
    'g': '--.',
    'h': '....',
    'i': '..',
    'j': '.---',
    'k': '-.-',
    'l': '.-..',
    'm': '--',
    'n': '-.',
    'o': '---',
    'p': '.--.',
    'q': '--.-',
    'r': '.-.',
    's': '...',
    't': '-',
    'u': '..-',
    'v': '...-',
    'w': '.--',
    'x': '-..-',
    'y': '-.--',
    'z': '--..'
}

def encode(word: str):
    return ''.join(MORSE_CODE[c] for c in word)

def wordBreakMorse(s: str, wordDict):
    # Same base-case semantics as LeetCode 139:
    # empty string is always segmentable (choose zero words)
    if not s:
        return True

    # non-empty string cannot be segmented if dictionary is empty
    if not wordDict:
        return False

    # Convert every dictionary word into its Morse string once.
    morse_words = [encode(word) for word in wordDict]

    # dp[i] == True  <=>  s[0:i] can be segmented
    dp = [False] * (len(s) + 1)
    dp[0] = True

    for i in range(1, len(dp)):
        for mw in morse_words:
            L = len(mw)
            if i >= L and s[i - L:i] == mw and dp[i - L]:
                dp[i] = True
                break

    return dp[-1]


## LC 3 — Longest substring without repeating **letters** (Morse)

**Classical LC3:** Longest substring with no repeated **characters** in a normal string.

**This variant:** `s` is a Morse stream (`'.'` / `'-'`). A **decoding** splits `s` into valid ITU Morse codewords (each codeword = one letter `a`–`z`). The split is **not unique** (same dots/dashes can mean different letter sequences).

**Task:** Over **every substring** of `s` and **every valid decoding** of that substring into letters, find the **maximum number of letters** in a decoding where **no letter repeats** (compare letters, not Morse symbols). Return that maximum (or `0` for empty `s`).

**Why it’s harder:** You cannot use a simple sliding window on `'.'`/`'-'`; you choose letter boundaries. Typical approach: DP on position + bitmask of which of the 26 letters you’ve used; allow a fresh start at every index so “any substring” is covered.

**Quick intuition:**
- `"...."` can decode as `"h"` (one letter) or `"es"` (two distinct letters) → answer **2** if that substring is allowed.
- Answer counts **decoded letters**, bounded by 26.

`def longestUniqueDecodedSubstringMorse(s: str) -> int`


In [ ]:
# Reuse MORSE_CODE from above.
# Build a reverse map: morse_string -> letter_index (0..25).
# Example: ".-" -> 0 ("a"), "-..." -> 1 ("b"), ...
MORSE_TO_IDX = {code: (ord(ch) - ord('a')) for ch, code in MORSE_CODE.items()}

def longestUniqueDecodedSubstringMorse(s: str) -> int:
    # dp[i][mask] = length of the longest substring ending at i with the given mask
    dp = [ {0:0} for i in range(len(s)+1)]
    
    for i in range(0, len(s)):
        for mask, length in dp[i].items():
            for L in range(1,5):
                if i+L > len(s):
                    break
                code = s[i:i+L]

                if code in MORSE_TO_IDX:
                    letter_idx = MORSE_TO_IDX[code]
                    new_mask = mask | (1 << letter_idx)

                    if (1 << letter_idx) & mask == 0: # same letter doesn't repeat
                        dp[i+L][new_mask] = max(dp[i+L].get(new_mask, 0), length + 1)
    
    out = 0
    for i in range(len(dp)):
        for mask, length in dp[i].items():
            out = max(out, length)
    return out

# Quick sanity checks
print(longestUniqueDecodedSubstringMorse(""))          # 0
print(longestUniqueDecodedSubstringMorse("...."))      # 2 ("es" or "se")
print(longestUniqueDecodedSubstringMorse(".--...."))   # 4 (take substring ".--..." -> "etni")
print(longestUniqueDecodedSubstringMorse(".-.-"))      # 3 ("eta")


## LC 1003 — Valid after substitutions (Morse)

**Classical [LC 1003](https://leetcode.com/problems/check-if-word-is-valid-after-substitutions/):** A string over `{a,b,c}` is **valid** if it is empty, or has the form  
`a + (valid) + b + (valid) + c`  
(recursive definition).

**This variant:** The three symbols in that rule are **not** single ASCII characters; they are the **fixed Morse strings** for the letters `a`, `b`, `c`:
- `T_a = M(a)`, `T_b = M(b)`, `T_c = M(c)` using your ITU table (next cell).
Concatenate with **no spaces**: e.g. the smallest nonempty abstract word is `"abc"`, which becomes one continuous bit pattern `T_a + T_b + T_c`.

**Task:** Given Morse bitstring `s`, return whether `s` belongs to that language (same recursion as 1003, but each `a`/`b`/`c` in the grammar is replaced by `T_a`, `T_b`, `T_c`).

**How to think about checking:** Same stack idea as classic 1003, but the stack holds `.`/`'-'`. After each push, if the **suffix** of the stack (last `len(T_a)+len(T_b)+len(T_c)` symbols) **exactly equals** `T_a + T_b + T_c`, delete that whole suffix. At the end, valid iff the stack is empty.

**Examples (abstract word → encode each letter with `MORSE_CODE`):**
- `""` → **True**
- `"abc"` → one layer → **True**
- `"aabcbc"` and `"abcabcababcc"` are standard 1003 **True** cases; encode to Morse and your function should still return **True**
- `"abac"` → **False** in 1003; same after Morse encoding

`def isValidAfterSubstitutionsMorse(s: str) -> bool`


In [3]:
# LC 1003 Morse — run a cell that defines MORSE_CODE (a..z) first, e.g. the Word Break cell.

T_A = MORSE_CODE["a"]
T_B = MORSE_CODE["b"]
T_C = MORSE_CODE["c"]
ABC = T_A + T_B + T_C  # one outer layer "a"+..."b"+..."c" in morse bits
_N = len(ABC)


def isValidAfterSubstitutionsMorse(s: str) -> bool:
    stack = []

    for c in s:
        stack.append(c)
        if len(stack) >= _N and stack[-_N:] == list(ABC):
            stack = stack[:-_N]

    return len(stack) == 0


def _morse_from_abcword(word: str) -> str:
    return "".join(MORSE_CODE[c] for c in word)


print(isValidAfterSubstitutionsMorse(""))
print(isValidAfterSubstitutionsMorse(_morse_from_abcword("abc")))
print(isValidAfterSubstitutionsMorse(_morse_from_abcword("aabcbc")))
print(isValidAfterSubstitutionsMorse(_morse_from_abcword("abac")))


True
True
True
False


LC91 Variant — Decode Ways (Morse Stream)
Problem
You are given a string s consisting only of '.' and '-'. It represents a continuous Morse stream with no separators between letters.

You are also given the standard ITU Morse mapping for lowercase English letters a–z:

Each letter ch maps to a Morse codeword M(ch) (a string of ./-), e.g.
M(e)=".", M(t)="-", M(a)=".-", M(h)="....", etc.
A decoding of s is any way to split s into contiguous chunks, where each chunk equals M(ch) for some letter ch.
Because Morse codewords have variable length and are not prefix-free (for example "." is e but ".." is i), the same stream can have multiple valid decodings.

Return the number of distinct decodings of the entire string s. Since the count can be large, return it modulo 
10
9
+
7
10 
9
 +7.

Function signature
def decodeWaysMorse(s: str) -> int:
    ...
Edge cases
If s == "", return 0 (no message to decode).
(In DP you’ll still typically use dp[0]=1 as a helper base case; the final API returns 0 for empty input.)
Examples
s = ".-"
Valid decodings:

".-" → "a"
"." + "-" → "e" + "t"
Output: 2

s = "...."
Valid decodings include:

"h" ("....")
"s" + "e" ("..." + ".")
"e" + "e" + "i" ("." + "." + "..")
"e" + "e" + "e" + "e" ("." + "." + "." + ".")
and others…
Output: 8

s = "--.."
"z" ("--..")
"m" + "i" ("--" + "..")
"t" + "b" ("-" + "-...") is not possible because s is "--..", not "-..." at the end.
Output: 2

Constraints (suggested)
0 <= len(s) <= 200000
s[i] in {'.', '-'}
Notes / intended DP insight (LC91-style)
Let dp[i] be the number of ways to decode prefix s[0:i]. Then:

dp[0] = 1
For each position i, try extending by a Morse codeword of length 1..4 (since ITU a–z lengths are at most 4):
if s[i:i+L] is a valid letter code, then dp[i+L] += dp[i]
Answer is dp[len(s)] (and return 0 if s is empty, per the edge-case rule above).

In [3]:
def decodeWaysMorse(s: str) -> int:
    """LC91-style count of decodings, but the input is a continuous Morse stream.

    We count the number of ways to split `s` into chunks, where each chunk is exactly
    the ITU Morse code for some letter a-z (using the global MORSE_CODE dict).

    Per the problem statement in this notebook: if s == "", return 0.
    """

    MOD = 10**9 + 7

    # Problem's requested API behavior.
    if s == "":
        return 0

    # All valid letter codewords (a..z). Each is length 1..4 in ITU mapping.
    codes = set(MORSE_CODE.values())

    n = len(s)

    # dp[i] = number of ways to decode prefix s[0:i]
    # Helper base-case: one way to decode empty prefix.
    dp = [0] * (n + 1)
    dp[0] = 1

    for i in range(n):
        if dp[i] == 0:
            continue

        # Try to take one letter of Morse length 1..4 starting at i.
        for L in range(1, 5):
            j = i + L
            if j > n:
                break

            if s[i:j] in codes:
                dp[j] = (dp[j] + dp[i]) % MOD

    return dp[n]


# Quick sanity checks
print(decodeWaysMorse(""))       # 0 (per spec)
print(decodeWaysMorse(".-"))     # 2: "a" or "e"+"t"
print(decodeWaysMorse("...."))   # 8
print(decodeWaysMorse("--.."))   # 2: "z" or "m"+"i"


0
2
8
8
